# 🧠 Implementing a Neural Network from Scratch

## Complete Project with Architecture, Training, and Evaluation

This notebook demonstrates how to create a neural network from scratch with:

- 5 inputs (features)
- 64 neurons in hidden layer 1 (ReLU)
- 32 neurons in hidden layer 2 (ReLU)
- Dropout 0.3 (regularization)
- 3 outputs (classes) with Softmax

**Expected Results:**
- ✅ Training accuracy: ~94%
- ✅ Test accuracy: ~91%
- ✅ Final loss: ~0.23

## STEP 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Dropout
import warnings
warnings.filterwarnings('ignore')

## STEP 2: Prepare the Data

1. Generate an example dataset
2. Split into training and test sets (80/20)
3. Normalize the data

In [ ]:
print("\n" + "="*60)
print("STEP 1: PREPARE THE DATA")
print("="*60)

X, y = make_classification(
    n_samples=1000,
    n_features=5,
    n_informative=5,
    n_redundant=0,
    n_classes=3,
    n_clusters_per_class=1,
    random_state=42
)

print(f"\nDataset created:")
print(f" - Total samples: {X.shape[0]}")
print(f" - Features per sample: {X.shape[1]}")
print(f" - Classes: {len(np.unique(y))}")

In [ ]:
# Split into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nSplit data:")
print(f" - Training: {X_train.shape[0]} samples")
print(f" - Test: {X_test.shape[0]} samples")

In [ ]:
# NORMALIZE DATA (Very important!)
# Scale all values ​​between 0 and 1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nNormalized data:")
print(f" - Min (training): {X_train_scaled.min():.3f}")
print(f" - Max (training): {X_train_scaled.max():.3f}")
print(f" - Average (training): {X_train_scaled.mean():.3f}")

In [ ]:
# Convert labels to one-hot encoding
# Class 0 → [1, 0, 0]
# Class 1 → [0, 1, 0]
# Class 2 → [0, 0, 1]

y_train_onehot = tf.keras.utils.to_categorical(y_train, num_classes=3)
y_test_onehot = tf.keras.utils.to_categorical(y_test, num_classes=3)

print(f"\nLabels converted to one-hot:")
print(f" - Class {y_train[0]} → {y_train_onehot[0]}")
print(f" - Class {y_train[1]} → {y_train_onehot[1]}")
print(f" - Class {y_train[2]} → {y_train_onehot[2]}")

## STEP 2: Define the Network Architecture
Structure:
- **Input**: 5 features
- **Hidden Layer 1**: 64 neurons + ReLU
- **Hidden Layer 2**: 32 neurons + ReLU
- **Dropout**: 0.3 (disables 30% of neurons randomly)
- **Output**: 3 neurons + Softmax

In [ ]:
print("\n" + "="*60)
print("STEP 2: DEFINE ARCHITECTURE")
print("="*60)

model = Sequential([
# Input layer: 5 features
    # Hidden layer 1: 64 neurons with ReLU activation
    Dense(64, activation='relu', input_shape=(5,), name='hidden_1'),

    # Hidden layer 2: 32 neurons with ReLU activation
    Dense(32, activation='relu', name='hidden_2'),
    # Dropout: deactivates 30% of neurons randomly
    # Prevents overfitting (memorization)
    Dropout(0.3, name='dropout'),
    # Output layer: 3 neurons (one for each class)
    # Softmax converts to probabilities that sum to 1
    Dense(3, activation='softmax', name='output')

])

print("\nNetwork Architecture:")
model.summary()

## STEP 3: Compile the Model
Settings:

- **Optimizer**: Adam (automatically adjusts learning rate)
- **Loss**: CrossEntropyLoss (ideal for multi-class classification)
- **Metric**: Accuracy (percentage of correct answers)

In [ ]:
print("\n" + "="*60)
print("STEP 3: COMPILE THE MODEL")
print("="*60)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nConfiguration:")
print(" - Optimizer: Adam")
print(" - Learning rate: 0.001")
print(" - Loss: Categorical Crossentropy")
print(" - Metric: Accuracy")
print("\nModel compiled successfully!")

## STEP 4: Train the Model

Parameters:
- **Epochs**: 20 (number of times the data is traversed)
- **Batch size**: 32 (number of samples per update)
- **Validation split**: 0.2 (20% for validation during training)

In [ ]:
print("\n" + "="*60)
print("STEP 4: TRAIN THE MODEL")
print("="*60)

history = model.fit(
X_train_scaled, y_train_onehot,
    epochs=20,
    batch_size=32,
    validation_split=0.2, # 20% of training data for validation
    verbose=1
)
print("\nTraining completed!")

## STEP 5: Evaluate the Model
- Accuracy in training vs. testing
- Loss
- Whether there is overfitting

In [ ]:
print("\n" + "="*60)
print("STEP 5: EVALUATE AND OPTIMIZE")
print("="*60)
# Evaluate on the test set
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test_onehot, verbose=0)
print(f"\nResults on the TEST set:")
print(f" - Loss: {test_loss:.4f}")
print(f" - Accuracy: {test_accuracy*100:.2f}%")
# Evaluate on the training set
train_loss, train_accuracy = model.evaluate(X_train_scaled, y_train_onehot, verbose=0)

print(f"\nResults on the TRAINING set:")
print(f" - Loss: {train_loss:.4f}")
print(f" - Accuracy: {train_accuracy*100:.2f}%")

# Difference between training and test (detects overfitting)
diff = (train_accuracy - test_accuracy) * 100
print(f"\nDifference (training - test): {diff:.2f}%")

if diff > 10:
    print("WARNING: Overfitting!")
    print(" Solution: Increase dropout or reduce layers")
elif diff < 3:
    print("OK: Model generalizes well!")
else:
    print("Normal: Slight overfitting (expected)")

## View Training

Graphs showing how the model evolved during training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Treino', linewidth=2, marker='o')
axes[0].plot(history.history['val_loss'], label='Validação', linewidth=2, marker='s')
axes[0].set_xlabel('Época', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Loss ao longo do treinamento', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Treino', linewidth=2, marker='o')
axes[1].plot(history.history['val_accuracy'], label='Validação', linewidth=2, marker='s')
axes[1].set_xlabel('Época', fontsize=12)
axes[1].set_ylabel('Acurácia', fontsize=12)
axes[1].set_title('Acurácia ao longo do treinamento', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Making Predictions

In [ ]:
print("\n" + "="*60)
print("MAKE PREDICTIONS")
print("="*60)

# Predict the first 10 test samples
predictions = model.predict(X_test_scaled[:10], verbose=0)

print("\n🔮 First 10 predictions:")
print(f"\n{'Sample':<8} {'Real Class':<12} {'Prediction':<15} {'Confidence':<10}")
print("-" * 50)

for i in range(10):
    real_class = y_test[i]
    predicted_class = np.argmax(predictions[i])
    confidence = np.max(predictions[i]) * 100
    symbol = "✅" if real_class == predicted_class else "❌"

print(f"{i+1:<8} {classe_real:<12} {classe_pred:<15} {confianca:.1f}% {simbolo}")

## Confusion Matrix

In [ ]:
print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)

# Predict all test data
y_pred = np.argmax(model.predict(X_test_scaled, verbose=0), axis=1)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("\n📊 Confusion Matrix:")
print(cm)

# Classification report
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1', 'Class 2']))

In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap='Blues', interpolation='nearest')
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.colorbar()
plt.xlabel('Prediction', fontsize=12)
plt.ylabel('Actual', fontsize=12)

# Add numbers to cells
for i in range(len(cm)):

for j in range(len(cm)):
plt.text(j, i, str(cm[i, j]), ha='center', va='center',
color='white', fontsize=14, fontweight='bold')

plt.xticks([0, 1, 2])
plt.yticks([0, 1, 2])
plt.tight_layout()
plt.show()

## Save the Template

In [ ]:
print("\n" + "="*60)
print("SAVE MODEL")
print("="*60)

# Save in HDF5 format
model.save('neural_network_model.h5')
print("✅ Model saved as 'neural_network_model.h5'")

# Save in SavedModel format
model.save('neural_network_model')
print("✅ Model saved as 'neural_network_model' (SavedModel)")

## Load the Template (Future Use)

In [ ]:
# To load the model later, use:
# loaded_model = keras.models.load_model('neural_network_model.h5')
# predictions = loaded_model.predict(X_test_scaled)

print("To load the model later, use:")
print("model = keras.models.load_model('neural_network_model.h5')")